# 📐 Class 2.1: Prompt Engineering, *Sprint 1: Write the Spec*
### GenAI & Agentic AI for SAP Professionals · Day 2 · Morning · 2 hours

**Project NOVA-AP:** Nova Industries' Accounts Payable shared mailbox receives **~300 vendor emails a day**.
At ~6 minutes each, that's **30 hours of manual reading per day (≈ 3.75 FTE)**. The steering committee wants AI to **triage and summarise** them.

**Your role:** the functional consultant. **Your deliverable:** the specification for the AI, which is **the prompt**.

| Phase | Topic | Cells |
|---|---|---|
| 1 · The requirement | A one-line requirement and what it delivers | ▶ 1 |
| 2 · The FS template | 5 sections of a good prompt: **R-T-C-F-R** | ▶ 2–5 |
| 3 · Impact analysis | Remove one section: what breaks? | ▶ 6 |
| 4 · Design patterns | Zero-shot · few-shot · stakeholder view | ▶ 7–10 |
| 5 · Build workshop | Hands-on + FS quality review | ▶ 11–14 |

**Runs locally on Ollama.** The model name lives in `bootcamp_config.json`.
Each slide shows a **▶ CELL** reference: switch here and run that cell. 🔒 Mock data only.

## ⚙️ ▶ CELL S: Setup
Checks that Ollama is running and the model is downloaded, then runs a connection test.

In [1]:
# ▶ CELL S · SETUP: connect to the local AI (Ollama). Run once.
import json, re, csv, time
from bootcamp_llm import *          # ask_ai, chat, ask_small, ask_json, show, parse_json, check_setup, last_usage, MODEL
from IPython.display import Markdown, display

DATA = json.load(open("../data/emails.json", encoding="utf-8"))
check_setup()
print("\n🤖", ask_ai("Reply with exactly these words: Connection OK", temperature=0, max_tokens=10),
      f"· {last_usage['seconds']} s · model = {MODEL}")

✅ Ollama 0.40.2 is running at http://localhost:11434
✅ Main model : gemma4:e2b
✅ Small model: gemma4:e2b
⚙️  Config: /Users/ra20586957wipro.com/Documents/Enchanté/Conversations/DC7FB73D-48DB-4ECB-884F-FD8CEBF33240/Day2_Prompt_Engineering_Ollama/bootcamp_config.json



🤖 Connection OK · 7.5 s · model = gemma4:e2b


---
# 1️⃣ The requirement
### ▶ CELL 1: The business user's request: *"Summarise this email."*
✋ *You're the AP team lead. Can you **act** on this output: escalate, assign, close?*

In [2]:
# ▶ CELL 1 · A one-line requirement
EMAIL_1 = DATA["EMAIL_1"]
print(EMAIL_1, "\n" + "─" * 70)

weak = ask_ai("Summarise this email.\n\n" + EMAIL_1, temperature=0)
show(weak)
print(f"\n⏱ {last_usage['seconds']} s · {len(weak.split())} words")

From: Rahul Verma, Bharat Logistics
Subject: Invoice INV-77821 - payment pending

Hi team,
We sent invoice INV-77821 for PO 4500012346 last month for ₹7,80,000.
Your AP team says the price is wrong, but we followed the rate card from
our last discussion. The goods were delivered to Plant 1000. This payment is
now 20 days overdue and is affecting our cash flow. Please check urgently
and let us know when we can expect payment, otherwise we may need to put
further deliveries on hold.
Regards,
Rahul 
──────────────────────────────────────────────────────────────────────


Here's a summary of the email:

**Rahul Verma from Bharat Logistics is urgently requesting payment for invoice INV-77821 amounting to ₹7,80,000, which is now 20 days overdue.** The recipient's Accounts Payable team is disputing the price, but Rahul insists the price was based on the agreed rate card from a previous discussion, and the goods were delivered to Plant 1000. Rahul warns that the delay is affecting their cash flow and threatens to **halt further deliveries** if payment is not resolved immediately.


⏱ 2.0 s · 82 words


> 📄 Readable, but **not actionable**: no PO number, no escalation decision, no owner, no next step.
> It's the same as a one-line requirement in a blueprint workshop: **the build was fine, but the spec was missing.**

---
# 2️⃣ The FS template: 5 sections, R-T-C-F-R
| FS section | Prompt part | Question | Our example |
|---|---|---|---|
| 1. User / persona | **R**ole | Who is the AI acting as? | SAP procurement analyst |
| 2. Requirement | **T**ask | What exactly must it do? | Summarise the dispute |
| 3. Business rules | **C**ontext | Which policies apply? | Above ₹5,00,000 → Finance Controller |
| 4. Output layout | **F**ormat | What structure? | Fixed fields |
| 5. Validations | **R**ules | What must it never do? | No guessing: "Not stated" |

### ▶ CELL 2: Write each section separately, like an FS document

In [3]:
# ▶ CELL 2 · The 5 FS sections
ROLE    = "You are an SAP procurement analyst."
TASK    = "Task: Summarise this invoice dispute email."
CONTEXT = ("Context (company policy):\n"
           "- ₹5 lakh = ₹5,00,000.\n"
           "- If the disputed invoice amount is ABOVE ₹5,00,000 → escalate to the Finance Controller.\n"
           "- Otherwise → the AP team handles it.")
FORMAT  = ("Format:\n- Vendor:\n- Invoice number:\n- PO number:\n- Invoice amount:\n- Issue (1 line):\n"
           "- Escalate to Finance Controller? (Yes/No + reason):\n- Next action:")
RULES   = ('Rules:\n- If a value is not in the email, write "Not stated". Do not guess.\n'
           "- Keep each line under 20 words.")

def spec_prompt(*sections, email=EMAIL_1):
    """Assemble the chosen FS sections, then attach the email between <<< >>> markers."""
    return "\n".join(sections) + "\n\nEmail:\n<<<\n" + email + "\n>>>"

print(spec_prompt(ROLE, TASK, CONTEXT, FORMAT, RULES))

You are an SAP procurement analyst.
Task: Summarise this invoice dispute email.
Context (company policy):
- ₹5 lakh = ₹5,00,000.
- If the disputed invoice amount is ABOVE ₹5,00,000 → escalate to the Finance Controller.
- Otherwise → the AP team handles it.
Format:
- Vendor:
- Invoice number:
- PO number:
- Invoice amount:
- Issue (1 line):
- Escalate to Finance Controller? (Yes/No + reason):
- Next action:
Rules:
- If a value is not in the email, write "Not stated". Do not guess.
- Keep each line under 20 words.

Email:
<<<
From: Rahul Verma, Bharat Logistics
Subject: Invoice INV-77821 - payment pending

Hi team,
We sent invoice INV-77821 for PO 4500012346 last month for ₹7,80,000.
Your AP team says the price is wrong, but we followed the rate card from
our last discussion. The goods were delivered to Plant 1000. This payment is
now 20 days overdue and is affecting our cash flow. Please check urgently
and let us know when we can expect payment, otherwise we may need to put
further deli

### ▶ CELL 3: Add the sections step by step
Same model, same email. Only the specification changes.

In [4]:
# ▶ CELL 3 · The spec, version by version
versions = {
    "v0.1 · Role + Task":                (ROLE, TASK),
    "v0.2 · + Output layout":            (ROLE, TASK, FORMAT),
    "v1.0 · + Business rules + Validations": (ROLE, TASK, CONTEXT, FORMAT, RULES),
}
for name, sections in versions.items():
    display(Markdown(f"### {name}"))
    show(ask_ai(spec_prompt(*sections), temperature=0))
    print(f"⏱ {last_usage['seconds']} s")

### v0.1 · Role + Task

## Invoice Dispute Summary for SAP Procurement Analyst

**Subject:** Dispute regarding Invoice INV-77821 (PO 4500012346)

**Sender:** Rahul Verma, Bharat Logistics

**Invoice Details:**
* **Invoice Number:** INV-77821
* **Purchase Order:** PO 4500012346
* **Amount:** ₹7,80,000
* **Goods Delivered To:** Plant 1000

**Issue:**
The vendor (Bharat Logistics) is disputing the invoice amount, claiming the price is incorrect. They assert that they adhered to the **rate card agreed upon during a previous discussion.**

**Urgency/Impact:**
* The payment is **20 days overdue**.
* The delay is negatively impacting the vendor's **cash flow**.
* The vendor has threatened to **suspend future deliveries** if payment is not resolved urgently.

**Action Required (For Procurement/AP Team):**
The AP team needs to urgently review the pricing discrepancy, referencing the agreed-upon rate card, and provide a clear timeline for payment.

⏱ 4.1 s


### v0.2 · + Output layout

Here is the summary of the invoice dispute email:

- **Vendor:** Bharat Logistics
- **Invoice number:** INV-77821
- **PO number:** 4500012346
- **Invoice amount:** ₹7,80,000
- **Issue (1 line):** Disagreement over the invoice price, with the vendor citing adherence to the agreed rate card. Payment is 20 days overdue.
- **Escalate to Finance Controller? (Yes/No + reason):** Yes + Due to potential disruption of future deliveries.
- **Next action:** Investigate the price discrepancy and confirm payment timeline.

⏱ 2.4 s


### v1.0 · + Business rules + Validations

- Vendor: Bharat Logistics
- Invoice number: INV-77821
- PO number: 4500012346
- Invoice amount: ₹7,80,000
- Issue (1 line): Disputed price based on rate card.
- Escalate to Finance Controller? Yes (Amount > ₹5,00,000)
- Next action: Finance Controller review.

⏱ 1.7 s


### ▶ CELL 4: Acceptance criteria. Does the output contain what the business needs?

In [5]:
# ▶ CELL 4 · Acceptance test: one-line requirement vs full spec
ACCEPTANCE = {"Invoice no.": "INV-77821", "PO no.": "4500012346", "Amount": "7,80,000",
              "Escalation": "Escalate", "Next action": "Next action"}

full = ask_ai(spec_prompt(ROLE, TASK, CONTEXT, FORMAT, RULES), temperature=0)

print(f"{'Criterion':14} | {'One-liner':9} | {'Full spec':9}")
print("─" * 40)
for label, needle in ACCEPTANCE.items():
    w = "✅" if needle.lower() in weak.lower() else "❌"
    f = "✅" if needle.lower() in full.lower() else "❌"
    print(f"{label:14} | {w:^9} | {f:^9}")

Criterion      | One-liner | Full spec
────────────────────────────────────────
Invoice no.    |     ✅     |     ✅    
PO no.         |     ❌     |     ✅    
Amount         |     ✅     |     ✅    
Escalation     |     ❌     |     ✅    
Next action    |     ❌     |     ✅    


### ▶ CELL 5: Business rules must be **explicit**, like configuration, not conversation

In [6]:
# ▶ CELL 5 · Vague rule vs explicit rule
VAGUE = "Context: Disputes on invoices above ₹5 lakh must go to the finance controller."
for name, ctx in [("😕 Vague", VAGUE), ("✅ Explicit", CONTEXT)]:
    answer = ask_ai(spec_prompt(ROLE, TASK, ctx, FORMAT, RULES), temperature=0)
    line = next((l for l in answer.splitlines() if "escalat" in l.lower()), "(no escalation line)")
    print(f"{name:11} → {line.strip()}")

😕 Vague     → - Escalate to Finance Controller? Yes + Price dispute.


✅ Explicit  → - Escalate to Finance Controller? Yes (Amount > ₹5,00,000)


> 🔎 **What we saw in testing (same model, same email):**
> - Run A: vague rule → *"Escalate? **No**: price dispute is below ₹5 lakh"* ❌ (₹7,80,000 is **above** ₹5 lakh)
> - Run B: vague rule → *"Escalate? **Yes**: price dispute"* ⚠️ (right answer, **wrong reason**)
> - Explicit rule → *"Escalate? **Yes (Amount > ₹5,00,000)**"* ✅ on every run
>
> 🔑 Write rules the way you'd **maintain a config table**: exact values, exact thresholds, an exact owner.
> "₹5 lakh = ₹5,00,000. Above it → Finance Controller. Otherwise → AP team."

---
# 3️⃣ Impact analysis
### ▶ CELL 6: Remove one FS section at a time. What breaks?
*Like a regression test after a change request.*

In [7]:
# ▶ CELL 6 · Remove one section, re-test
sections = {"Role": ROLE, "Task": TASK, "Context": CONTEXT, "Format": FORMAT, "Rules": RULES}
for missing in sections:
    kept = [v for k, v in sections.items() if k != missing]
    answer = ask_ai(spec_prompt(*kept), temperature=0)
    score = sum(n.lower() in answer.lower() for n in ACCEPTANCE.values())
    print(f"Without {missing:8} → {score}/{len(ACCEPTANCE)} criteria met · {len(answer.split()):>3} words")

Without Role     → 5/5 criteria met ·  42 words


Without Task     → 5/5 criteria met ·  37 words


Without Context  → 5/5 criteria met ·  47 words


Without Format   → 3/5 criteria met ·  44 words


Without Rules    → 5/5 criteria met ·  71 words


> 📉 Typically **Output layout (Format)** is the one you can't skip. Without it, the structure changes from run to run.

---
# 4️⃣ Design patterns
### ▶ CELL 7: Pattern 1, zero-shot (the requirement only)

In [8]:
# ▶ CELL 7 · Zero-shot: no examples
question = DATA["CLASSIFY_SET"]["C"]
show(ask_ai(f'Classify this email: "{question}"', temperature=0))

This email can be classified as **Informational Request** or **Business Inquiry**.

Here's a breakdown of why and other ways to look at it:

**Primary Classification: Informational Request**

* **Reason:** The sender is explicitly asking for specific data (the "current unit price") needed for a business decision or planning.

**Secondary Classifications:**

* **Business Inquiry:** It's a direct question related to procurement, pricing, or supply chain management.
* **Transactional (Implied):** While it's not a direct transaction, it's a necessary step *before* a transaction (like placing an order).
* **Operational:** It relates directly to the ongoing processes of the business (pricing, inventory, purchasing).

**In summary, the most accurate classification is:** **Informational Request (related to Procurement/Pricing)**

😕 Its own categories and an essay, so nothing a routing rule can use.

### ▶ CELL 8: Pattern 2, few-shot (sample data in the spec)

In [9]:
# ▶ CELL 8 · Few-shot: 3 reference examples
FEW_SHOT = """Classify each email into exactly one category:
INVOICE_ISSUE, DELIVERY_DELAY, PRICE_QUERY, OTHER.
Reply with the category only.

Email: "The goods for PO 4500012345 haven't arrived yet." → DELIVERY_DELAY
Email: "Invoice 9001 shows a higher price than agreed." → INVOICE_ISSUE
Email: "What is your price for 500 units of bearings?" → PRICE_QUERY

Email: "{email}" →"""

print(ask_ai(FEW_SHOT.format(email=question), temperature=0, max_tokens=10))

PRICE_QUERY


### ▶ CELL 9: Unit test, 5 emails scored against the expected results

In [10]:
# ▶ CELL 9 · Few-shot on 5 emails, auto-scored
score = 0
for k, email in DATA["CLASSIFY_SET"].items():
    label = ask_ai(FEW_SHOT.format(email=email), temperature=0, max_tokens=10).strip().strip("*.` ")
    ok = label == DATA["CLASSIFY_KEY"][k]
    score += ok
    print(f"{k}: {label:15} {'✅' if ok else '❌ expected ' + DATA['CLASSIFY_KEY'][k]}   {email[:55]}…")
print(f"\n🏁 Result: {score}/5 passed")

A: DELIVERY_DELAY  ✅   The goods for PO 4500012349 were due last week and stil…
B: INVOICE_ISSUE   ✅   Invoice INV-61002 shows ₹520 per unit, but our PO says …


C: PRICE_QUERY     ✅   Could you confirm the current unit price for material M…
D: OTHER           ✅   Please update our bank details for future payments; new…


E: DELIVERY_DELAY  ✅   Our truck broke down; the bearings for PO 4500012347 wi…

🏁 Result: 5/5 passed


> 🚩 **Email D:** *"Please update our bank details"*. It's labelled OTHER, but it's a classic **payment-fraud pattern**. Should an AI ever act on it alone? (Day 4: guardrails.)

### ▶ CELL 10: Pattern 3, stakeholder view (persona)

In [11]:
# ▶ CELL 10 · One email, three stakeholder outputs
EMAIL_2 = DATA["EMAIL_2"]
stakeholders = {
    "🧑‍💼 Procurement analyst (internal)": "You are an SAP procurement analyst. Summarise for the internal team in 3 bullets.",
    "🤝 Vendor relations (external reply)": "You are a polite vendor relations manager. Draft a short reply to the vendor (max 5 lines).",
    "📊 CFO (one-line status)":            "You are writing for a CFO. Give a 1-line update.",
}
for name, instruction in stakeholders.items():
    display(Markdown(f"#### {name}"))
    show(ask_ai(instruction + "\n\n<<<\n" + EMAIL_2 + "\n>>>", temperature=0))

#### 🧑‍💼 Procurement analyst (internal)

Here is the summary for the internal team:

* **Invoice Blocked:** Invoice INV-55190 for PO 4500012348 (amounting to ₹12,00,000) is currently blocked.
* **Reason for Block:** The block is due to a price variance, specifically related to the steel surcharge communicated in August.
* **Action Required:** Procurement needs to confirm the revised price to lift the block on the invoice.

#### 🤝 Vendor relations (external reply)

Dear Lakshmi,

Thank you for reaching out regarding PO 4500012348.

I will immediately check with our procurement team regarding the price variance and the steel surcharge.

We aim to resolve this swiftly. I will update you shortly.

Best regards,
[Your Name]

#### 📊 CFO (one-line status)

**Invoice for Gearbox order is blocked pending procurement confirmation on the revised price due to steel surcharge.**

---
# 5️⃣ Build workshop (40 min)
Change only the lines marked **← change me**, then press **Shift + Enter**.

### ▶ CELL 11: Task 1, write a full spec for Email 3 (a quantity difference)

In [12]:
# ▶ CELL 11 · Your spec
EMAIL_3 = DATA["EMAIL_3"]
print(EMAIL_3, "\n" + "─" * 70)

my_role    = "You are an SAP accounts payable clerk."                               # ← change me
my_task    = "Task: Summarise this email and recommend an action."                  # ← change me
my_context = "Context: Quantity differences of more than 2% need a credit note."    # ← change me
my_format  = "Format:\n- Vendor:\n- Invoice:\n- Difference:\n- Credit note needed (Yes/No):\n- Next action:"   # ← change me
my_rules   = 'Rules: If a value is missing, write "Not stated". Max 15 words per line.'  # ← change me

show(ask_ai(spec_prompt(my_role, my_task, my_context, my_format, my_rules, email=EMAIL_3), temperature=0))

From: Arjun Mehta, Delta Plastics
Subject: Small invoice query

Hi,
Invoice INV-30117 for PO 4500012353 (₹1,20,000) shows 400 housings, but our
delivery note says 380 were accepted at Plant 2000. Shall we issue a credit note
for the 20 rejected pieces?
Regards, Arjun 
──────────────────────────────────────────────────────────────────────


- Vendor: Delta Plastics
- Invoice: INV-30117
- Difference: 20 housings (approx 1.67%)
- Credit note needed (Yes): Yes
- Next action: Issue credit note for 20 rejected pieces.

✅ *Verify with Python:* 20 of 400 pieces = **5%** → above 2% → credit note **Yes**.
> 🧮 In our test run the model wrote *"approx 1.67%"*: **wrong calculation, right decision**. Never accept AI arithmetic unverified (Class 2.2).

### ▶ CELL 12: Task 2, design your own categories (few-shot) for the SAP support desk

In [13]:
# ▶ CELL 12 · Your own few-shot classifier
MY_FEW_SHOT = """Classify the ticket as: MASTER_DATA, AUTHORISATION, PERFORMANCE, OTHER. Reply with the label only.
Ticket: "Please create vendor V500 in company code 1000." → MASTER_DATA
Ticket: "I get 'No authorisation for transaction ME21N'." → AUTHORISATION
Ticket: "{t}" →"""                                                              # ← change the labels/examples

my_tickets = ["Report ZMM_STOCK takes 20 minutes to run.",                      # ← change me
              "Need access to MIGO for plant 2000.",
              "Change the bank key for vendor V300."]
for t in my_tickets:
    print(f"{ask_ai(MY_FEW_SHOT.format(t=t), temperature=0, max_tokens=10).strip():15} ← {t}")

PERFORMANCE     ← Report ZMM_STOCK takes 20 minutes to run.
AUTHORISATION   ← Need access to MIGO for plant 2000.


MASTER_DATA     ← Change the bank key for vendor V300.


### ▶ CELL 13: Task 3, a different stakeholder view

In [14]:
# ▶ CELL 13 · Your stakeholder
my_persona = "You are a strict internal auditor. List 3 risks in this email, one line each."   # ← change me
show(ask_ai(my_persona + "\n\n<<<\n" + EMAIL_2 + "\n>>>", temperature=0))

Here are 3 risks identified in the email:

1. **Invoice payment blockage:** The invoice is currently blocked, risking delayed payment.
2. **Unconfirmed price variance:** The justification for the price increase is not formally confirmed, risking financial discrepancy.
3. **Lack of clear escalation path:** The request relies on an unspecified party (procurement) for resolution, risking process inefficiency.

### ▶ CELL 14: Task 4, FS quality review (the AI scores your prompt)

In [15]:
# ▶ CELL 14 · Quality gate: score a prompt 0–2 per section
prompt_to_review = "Summarise this email."                                       # ← paste your spec here
show(ask_ai(f"""Score this prompt 0-2 on each of: Role, Task, Context, Format, Rules (max 10).
Show a small markdown table, then give ONE concrete improvement.
Prompt:
<<<
{prompt_to_review}
>>>""", temperature=0))

Here is the scoring and feedback for the prompt: "Summarise this email."

| Criterion | Score (0-2) | Justification |
|---|---|---|
| **Role** | 1 | No specified role for the summarizer (e.g., executive assistant, journalist). |
| **Task** | 2 | The task is clear: "Summarise." |
| **Context** | 0 | Missing crucial context: *Which* email? What is the email about? |
| **Format** | 1 | No required output format specified (e. e.g., bullet points, paragraph, length). |
| **Rules** | 1 | No constraints provided (e. e.g., tone, length limit, focus areas). |

**Concrete Improvement:**

Specify the **length and focus** required for the summary. For example: "Summarise this email into three bullet points, focusing only on the action items."

---
## ✅ Sprint 1 review: Class 2.1 takeaways
1. A prompt is a **functional specification**: Role · Task · Context · Format · Rules.
2. Write business rules like **configuration**: exact numbers, thresholds and owners.
3. **Few-shot** = reference examples for consistent categories. **Persona** = the right output for each stakeholder.

> 🌉 **Sprint 2 (after lunch):** the output must feed a **system**, the logic must be **tested**, and the design must pass a **security review**. 🔐

---
## 🛠️ Trainer troubleshooting (Ollama)
| Message | Cause | Fix |
|---|---|---|
| `Cannot reach Ollama at http://localhost:11434` | Ollama isn't running | Windows: Start menu → **Ollama**. Mac: open the Ollama app. Or in a terminal: `ollama serve` |
| `Model '…' is not downloaded` | The model hasn't been pulled | In a terminal: `ollama pull <model name>` |
| `ModuleNotFoundError: bootcamp_llm` | The notebook was opened from a different folder | Open the **whole bundle folder** in VS Code or Antigravity, and keep the notebook inside `notebooks/` |
| Very slow first answer | The model is loading into memory | Normal. The next answers are faster. |
| Slow on every answer | The laptop is too small for the model | Choose a smaller `model` in `bootcamp_config.json` (e.g. `llama3.2:1b`), then **Restart kernel** |
| Answers differ from the slides | A different model or a new run | Expected: small local models vary. Discuss *why*. |
| Change the model mid-class | | Edit `bootcamp_config.json` → **Restart kernel** → run CELL S again |